# PolyTg-Bench exploratory analysis

This notebook inspects generated artifacts only. Production preprocessing, splitting, training, and evaluation remain in the command-line pipeline. Run `scripts/prepare_data.py` and benchmark experiments before using these cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
PROCESSED = ROOT / 'data' / 'processed' / 'polytg_cleaned.csv'
RESULTS = ROOT / 'results' / 'benchmark_results.csv'
print(f'Project root: {ROOT}')

## Dataset audit

Confirm row count, schema, missingness, target distribution, and duplicate canonical structures before interpreting any model comparison.

In [ ]:
if not PROCESSED.exists():
    raise FileNotFoundError('Run scripts/prepare_data.py before this notebook.')
data = pd.read_csv(PROCESSED)
summary = {
    'rows': len(data),
    'columns': list(data.columns),
    'missing_by_column': data.isna().sum().to_dict(),
    'duplicate_canonical_structures': int(data.duplicated('canonical_smiles').sum()),
}
summary

In [ ]:
target_column = 'tg'  # Match the active YAML configuration.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data[target_column].hist(ax=axes[0], bins=30)
axes[0].set(title='Tg distribution', xlabel='Tg', ylabel='Count')
data['canonical_smiles'].str.len().hist(ax=axes[1], bins=30)
axes[1].set(title='Canonical SMILES length', xlabel='Characters', ylabel='Count')
fig.tight_layout()

## Completed-run comparison

This view is descriptive. Use `scripts/generate_report.py` for confidence intervals and paired bootstrap comparisons on matched test rows.

In [ ]:
if not RESULTS.exists():
    raise FileNotFoundError('Run benchmark experiments before comparing results.')
results = pd.read_csv(RESULTS)
results.groupby(['representation', 'model', 'split_strategy']).agg(
    runs=('experiment_id', 'count'),
    mae_mean=('mae', 'mean'),
    mae_std=('mae', 'std'),
    rmse_mean=('rmse', 'mean'),
    r2_mean=('r2', 'mean'),
    training_seconds_mean=('training_time_seconds', 'mean'),
).reset_index()

## Error inspection

Load one saved prediction file from the results table and inspect its largest test errors. Keep molecular and measurement provenance in view when diagnosing outliers.

In [ ]:
selected = results.sort_values('mae').iloc[0]
prediction_path = Path(selected['prediction_file'])
if not prediction_path.is_absolute():
    prediction_path = ROOT / prediction_path
predictions = pd.read_csv(prediction_path)
test = predictions.loc[predictions['partition'] == 'test'].copy()
test['absolute_error'] = test['residual'].abs()
test.nlargest(15, 'absolute_error')[
    ['smiles', 'canonical_smiles', 'actual', 'predicted', 'residual', 'absolute_error']
]